In [0]:
import requests
import time
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, BooleanType

API_KEY = ""
BASE_URL = "https://rebrickable.com/api/v3/lego/colors/"
PAGE_SIZE = 1000
REQUEST_DELAY_SECONDS = 1.1  # stay within ~1 req/sec rate limit

In [0]:
def fetch_all_colors(api_key: str, base_url: str, page_size: int, delay: float) -> list:
    headers = {"Authorization": f"key {api_key}"}
    params = {"page": 1, "page_size": page_size}
    all_results = []

    next_url = base_url

    while next_url:
        response = requests.get(next_url, headers=headers, params=params, timeout=30)
        response.raise_for_status()

        data = response.json()
        all_results.extend(data.get("results", []))

        next_url = data.get("next")  # None when there are no more pages
        params = {}  # next_url already contains all query params

        print(f"Fetched {len(all_results)} / {data['count']} records")

        if next_url:
            time.sleep(delay)

    return all_results


colors_data = fetch_all_colors(API_KEY, BASE_URL, PAGE_SIZE, REQUEST_DELAY_SECONDS)
print(f"Total colors fetched: {len(colors_data)}")

In [0]:
schema = StructType([
    StructField("id",          IntegerType(), nullable=False),
    StructField("name",        StringType(),  nullable=True),
    StructField("rgb",         StringType(),  nullable=True),
    StructField("is_trans",    BooleanType(), nullable=True),
    StructField("external_ids", StringType(), nullable=True),  # stored as JSON string
])

# Flatten external_ids to a JSON string so it fits a simple schema
import json

rows = [
    (
        record["id"],
        record.get("name"),
        record.get("rgb"),
        record.get("is_trans"),
        json.dumps(record.get("external_ids")) if record.get("external_ids") else None,
    )
    for record in colors_data
]

spark = SparkSession.builder.getOrCreate()
df_colors = spark.createDataFrame(rows, schema=schema)

df_colors.printSchema()
df_colors.display(10, truncate=False)

In [0]:
from pyspark.sql.functions import current_timestamp, lit

df_colors_out = (
    df_colors
    .withColumn("_load_datetime",  current_timestamp())
    .withColumn("_record_source",  lit("API"))
)

display(df_colors_out)

In [0]:
# Write the DataFrame to an external volume as Parquet
# Update the path below to match your Unity Catalog external volume mount point
EXTERNAL_VOLUME_PATH = "/Volumes/lego_brickbase/bronze/raw_data_volume/colors"

df_colors.write \
    .mode("overwrite") \
    .parquet(EXTERNAL_VOLUME_PATH)

print(f"Colors written to: {EXTERNAL_VOLUME_PATH}")

In [0]:
from datetime import datetime
from pyspark.sql.functions import current_timestamp, lit

# Generate timestamp components for partitioned path and filename
now = datetime.utcnow()
year  = now.strftime("%Y")
month = now.strftime("%m")
day   = now.strftime("%d")
ts    = now.strftime("%Y%m%d_%H%M%S")

# Add audit columns
df_colors_out = (
    df_colors
    .withColumn("_load_datetime",  current_timestamp())
    .withColumn("_record_source",  lit("API"))
)

# Update the base path below to match your Unity Catalog external volume mount point
EXTERNAL_VOLUME_BASE = "/Volumes/lego_brickbase/bronze/raw_data_volume/colors"
PARTITION_PATH       = f"{EXTERNAL_VOLUME_BASE}/{year}/{month}/{day}/{ts}"
TEMP_PATH            = f"{PARTITION_PATH}/_tmp"
FILE_NAME            = f"raw_colors_{ts}.parquet"
FINAL_PATH           = f"{PARTITION_PATH}/{FILE_NAME}"

# Write as a single Parquet file to a temp staging directory
df_colors.coalesce(1) \
    .write \
    .mode("overwrite") \
    .parquet(TEMP_PATH)

# Locate the single part file and rename it to the desired filename
part_files = [f.path for f in dbutils.fs.ls(TEMP_PATH) if f.name.startswith("part-")]
dbutils.fs.mv(part_files[0], FINAL_PATH)
dbutils.fs.rm(TEMP_PATH, recurse=True)

print(f"Colors written to: {FINAL_PATH}")
